In [ ]:
import re
import subprocess

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats

plt.style.use('bmh')

PYTHON_PATH = '/home/andreas/mambaforge/envs/symdel/bin/python'
WORKER_SCRIPT = 'tmp/symscan_memory_worker.py'
SEQ_FILE = '../data/emerson_sequences.txt'
MAX_DISTANCE = 1
DISTANCE_TYPE = 'levenshtein'
N_REPS = 1

In [ ]:
!mkdir -p tmp

In [ ]:
df = pd.concat([
    pd.read_csv(f'../data/emerson_rep{i}.zip', sep=',', compression='zip')
    for i in range(1, 15)
    ])
seqs = df['cdr3'].drop_duplicates().sample(frac=1, random_state=0).reset_index(drop=True)
len(seqs)

In [ ]:
with open(SEQ_FILE, 'w') as f:
    f.write('\n'.join(seqs) + '\n')

In [ ]:
%%writefile tmp/symscan_memory_worker.py
# Standalone worker run as a fresh subprocess for each measurement, so that the peak RSS
# reported by `/usr/bin/time -v` reflects a single symscan.get_neighbors_within call in
# isolation rather than a cumulative/high-water mark across many calls in one process.
import sys

import symscan


def main():
    seq_file, n_sequence, max_distance, distance_type = sys.argv[1:5]
    n_sequence = int(n_sequence)
    max_distance = int(max_distance)

    with open(seq_file) as f:
        seqs = [next(f).strip() for _ in range(n_sequence)]

    symscan.get_neighbors_within(seqs, max_distance=max_distance, distance_type=distance_type)


if __name__ == '__main__':
    main()

In [ ]:
def measure_peak_memory_gb(n_sequence, max_distance=MAX_DISTANCE, distance_type=DISTANCE_TYPE):
    cmd = ['/usr/bin/time', '-v', PYTHON_PATH, WORKER_SCRIPT,
           SEQ_FILE, str(n_sequence), str(max_distance), distance_type]
    result = subprocess.run(cmd, capture_output=True, text=True)
    match = re.search(r'Maximum resident set size \(kbytes\): (\d+)', result.stderr)
    if match is None:
        raise RuntimeError(result.stderr)
    return int(match.group(1)) / 1024**2

sizes = np.unique(np.geomspace(1e5, len(seqs), num=12, dtype=int))
sizes

In [ ]:
rows = []
for n_sequence in sizes:
    for rep in range(N_REPS):
        memory_gb = measure_peak_memory_gb(n_sequence)
        rows.append({'algorithm': 'symscan', 'n_sequence': n_sequence,
                      'distance': MAX_DISTANCE, 'measure': DISTANCE_TYPE,
                      'memory_gb': memory_gb})
        print(n_sequence, rep, memory_gb)

mem_df = pd.DataFrame(rows)
mem_df.to_csv('../data/symscan_memory_benchmark.csv')
mem_df.groupby('n_sequence')['memory_gb'].agg(['mean', 'size'])

In [ ]:
mean = mem_df.groupby('n_sequence')['memory_gb'].mean()
x, y = mean.index.values, mean.values

slope, intercept, r, p, se = scipy.stats.linregress(np.log(x), np.log(y))
print(f'scaling exponent: {slope:.3} +/- {se:.2}')

fig, ax = plt.subplots(figsize=(3.4, 2.4))
ax.plot(x, y, 'o', label='SymScan')
ax.plot(x, np.exp(slope * np.log(x) + intercept), '-', color='C0')
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('# Sequences')
ax.set_ylabel('Peak memory (GB)')
fig.tight_layout(pad=0.0)
fig.savefig('figs/symscan_memory_benchmark.svg')